# Egyptian Legal AI — Automated Code & Test Audit

This notebook converts the diagnostic/import checks and distinct pytest targets found in `CODES.md` into a repeatable report.

**Run from the project root or set `PROJECT_ROOT` in the first code cell.** The notebook does not install packages or modify source files. Pytest tests can still modify test databases/files according to the project's own fixtures.

In [1]:
from pathlib import Path
import os, sys, subprocess, traceback, contextlib, io, time, json, re
from datetime import datetime
from collections import Counter


# Set this explicitly if the notebook is not launched from the project root:
PROJECT_ROOT = Path.cwd()  # Example: Path("/home/user/Egyptian-Legal-AI")

# Walk upward until we find a likely project root.
if not ((PROJECT_ROOT / "src").is_dir() and (PROJECT_ROOT / "pyproject.toml").exists()):
    candidates = [p for p in [PROJECT_ROOT, *PROJECT_ROOT.parents]
                  if (p / "src").is_dir() and (p / "pyproject.toml").exists()]
    if candidates:
        PROJECT_ROOT = candidates[0]

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(
        f"Could not locate project root. Current path: {Path.cwd()}. "
        "Set PROJECT_ROOT to the directory containing src/ and pyproject.toml."
    )

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + os.environ.get("PYTHONPATH", "")

print(f"Project root : {PROJECT_ROOT}")
print(f"Python       : {sys.version.split()[0]}")
print(f"Executable   : {sys.executable}")
print(f"Working dir  : {Path.cwd()}")

Project root : /run/media/ahmed/Data/RAG/Egyptian-Legal-AI
Python       : 3.11.17
Executable   : /home/ahmed/anaconda3/envs/genai/bin/python
Working dir  : /run/media/ahmed/Data/RAG/Egyptian-Legal-AI


## 1. Define checks

Import checks are executed inside the notebook process with captured output and full tracebacks on failure. Pytest suites are launched as subprocesses so their exit codes and output are preserved.

Repeated diagnostic variants in `CODES.md` that run the same pytest target only to change `tail`, `grep`, `head`, or output redirection are consolidated into one test. The source line references are retained.

In [2]:
# Import/code checks identified in CODES.md.
# Each check is independent: a failure will not stop later checks.
IMPORT_TESTS = [
    {
        "id": "IMP-01", "source_line": 318, "name": "Settings configuration import",
        "code": """from src.config.settings import settings\nprint(settings.LOG_LEVEL)\nprint("DATABASE_URL configured:", bool(settings.DATABASE_URL))\nprint(settings.DOCUMENT_STORAGE_DIR)"""
    },
    {
        "id": "IMP-02", "source_line": 3396, "name": "Storage, security, and service public imports",
        "code": """from src.storage.database.database import Database, get_database
from src.storage.database.models import Base, DocumentRecord, ConversationRecord, UserRecord
from src.storage.database.repositories import DocumentRepository, ConversationRepository
from src.storage.document_store import LocalDocumentStore
from src.storage.cache import RedisCache, NoOpCache, get_cache
from src.security import AuthContext, PasswordHasher, FileValidator, Permission, TenantContext
from src.services import DocumentService, ConversationService, IngestionService, SearchService, ChatService
print("All imports OK")"""
    },
    {
        "id": "IMP-03", "source_line": 3411, "name": "Database module import",
        "code": "from src.storage.database.database import Database"
    },
    {
        "id": "IMP-04", "source_line": 3415, "name": "SQLAlchemy dependency import",
        "code": "from sqlalchemy import create_engine\nprint('sqlalchemy OK')"
    },
    {
        "id": "IMP-05", "source_line": 3425, "name": "Database models import",
        "code": """from src.storage.database.database import Database
from src.storage.database.models import Base, DocumentRecord, ConversationRecord, UserRecord
print("Models OK")"""
    },
    {
        "id": "IMP-06", "source_line": 3433, "name": "Services and security imports",
        "code": """from src.services import DocumentService, ConversationService, IngestionService, SearchService, ChatService
from src.security import AuthContext, PasswordHasher, FileValidator, Permission
print("All imports OK")"""
    },
    {
        "id": "IMP-07", "source_line": 3443, "name": "DocumentService public import",
        "code": "from src.services import DocumentService"
    },
    {
        "id": "IMP-08", "source_line": 3447, "name": "IngestionService module import",
        "code": "from src.services.ingestion_service import IngestionService"
    },
    {
        "id": "IMP-09", "source_line": 3661, "name": "Services and security imports (repeat checkpoint)",
        "code": """from src.services import DocumentService, ConversationService, IngestionService, SearchService, ChatService
from src.security import AuthContext, PasswordHasher, FileValidator, Permission
print("All imports OK")"""
    },
    {
        "id": "IMP-10", "source_line": 3669, "name": "ChatService import",
        "code": """from src.services.chat_service import ChatService
print("ChatService OK")"""
    },
    {
        "id": "IMP-11", "source_line": 3678, "name": "RAGPipeline import",
        "code": "from src.rag.pipeline import RAGPipeline\nprint('OK')"
    },
    {
        "id": "IMP-12", "source_line": 3682, "name": "RAGError import",
        "code": "from src.domain.exceptions import RAGError\nprint('OK')"
    },
    {
        "id": "IMP-13", "source_line": 3686, "name": "Core services module imports",
        "code": """from src.services.chat_service import ChatService
from src.services.ingestion_service import IngestionService
from src.services.search_service import SearchService
print("All services OK")"""
    },
]

# Distinct pytest targets from CODES.md. Output-format-only retries were deduplicated.
PYTEST_TESTS = [

    # ============================================================
    # Phase 1 — Foundation & Configuration
    # ============================================================
    {
        "id": "PYT-01",
        "name": "Phase 1 — Foundation & Configuration",
        "args": [
            "tests/unit/test_settings.py",
            "tests/unit/test_file_utils.py",
            "tests/unit/test_text_utils.py",
            "-v", "--tb=short"
        ]
    },

    # ============================================================
    # Phase 2 — Domain Models
    # ============================================================
    {
        "id": "PYT-02",
        "name": "Phase 2 — Domain Models",
        "args": [
            "tests/unit/test_documents.py",
            "tests/unit/test_chunks.py",
            "-v", "--tb=short"
        ]
    },

    # ============================================================
    # Phase 3 — Document Ingestion
    # ============================================================
    {
        "id": "PYT-03",
        "name": "Phase 3 — Document Ingestion",
        "args": [
            "tests/unit/test_pdf_loader.py",
            "tests/unit/test_docx_loader.py",
            "tests/unit/test_loader_factory.py",
            "tests/unit/test_document_registry.py",
            "-v", "--tb=short"
        ]
    },

    # ============================================================
    # Phase 4 — Preprocessing
    # ============================================================
    {
        "id": "PYT-04",
        "name": "Phase 4 — Preprocessing",
        "args": [
            "tests/unit/test_arabic_normalizer.py",
            "tests/unit/test_legal_structure.py",
            "tests/unit/test_text_cleaner.py",
            "tests/unit/test_preprocessing_pipeline.py",
            "-v", "--tb=short"
        ]
    },

    # ============================================================
    # Phase 5–6 — Chunking & Embeddings
    # ============================================================
    {
        "id": "PYT-05",
        "name": "Phase 5–6 — Chunking & Embeddings",
        "args": [
            "tests/unit/test_chunking_and_embeddings.py",
            "-v", "--tb=short"
        ]
    },

    # ============================================================
    # Phase 7 — Vector Storage
    # ============================================================
    {
        "id": "PYT-06",
        "name": "Phase 7 — Vector Storage",
        "args": [
            "tests/unit/test_vector_store.py",
            "-v", "--tb=short"
        ]
    },

    # ============================================================
    # Phase 8 — Retrieval
    # ============================================================
    {
        "id": "PYT-07",
        "name": "Phase 8 — Retrieval",
        "args": [
            "tests/unit/test_retrieval.py",
            "-v", "--tb=short"
        ]
    },

    # ============================================================
    # Phase 9 — LLM & Generation
    # ============================================================
    {
        "id": "PYT-08",
        "name": "Phase 9 — LLM & Generation",
        "args": [
            "tests/unit/test_llm_providers.py",
            "tests/unit/test_responses.py",
            "tests/unit/test_model_config.py",
            "-v", "--tb=short"
        ]
    },

    # ============================================================
    # Phase 10 — RAG Pipeline
    # ============================================================
    {
        "id": "PYT-09",
        "name": "Phase 10 — RAG Pipeline",
        "args": [
            "tests/unit/test_rag_pipeline.py",
            "-v", "--tb=short"
        ]
    },

    # ============================================================
    # Phase 11 — Storage
    # ============================================================
    {
        "id": "PYT-10",
        "name": "Phase 11 — Storage",
        "args": [
            "tests/unit/test_database.py",
            "tests/unit/test_document_repository.py",
            "tests/unit/test_conversation_repository.py",
            "tests/unit/test_document_store.py",
            "tests/unit/test_cache.py",
            "-v", "--tb=short"
        ]
    },

    # ============================================================
    # Phase 12 — Services & Security
    # ============================================================
    {
        "id": "PYT-11",
        "name": "Phase 12 — Services & Security",
        "args": [
            "tests/unit/test_authentication.py",
            "tests/unit/test_authorization.py",
            "tests/unit/test_file_validation.py",
            "tests/unit/test_tenant_isolation.py",
            "tests/unit/test_document_service.py",
            "tests/unit/test_conversation_service.py",
            "-v", "--tb=short"
        ]
    },

    # ============================================================
    # Integration Tests — Phases 1–4
    # ============================================================
    {
        "id": "PYT-12",
        "name": "Integration — Phases 1–4 Workflow",
        "args": [
            "tests/integration/test_phase_01_to_04_workflow.py",
            "-v", "--tb=short"
        ]
    },

    # ============================================================
    # Integration Tests — Phases 3–6
    # ============================================================
    {
        "id": "PYT-13",
        "name": "Integration — Phases 3–6",
        "args": [
            "tests/integration/test_phase3_to_phase6.py",
            "-v", "--tb=short"
        ]
    },

    # ============================================================
    # Integration Tests — Phases 7–8
    # ============================================================
    {
        "id": "PYT-14",
        "name": "Integration — Phases 7–8",
        "args": [
            "tests/integration/test_phase7_8_integration.py",
            "-v", "--tb=short"
        ]
    },

    # ============================================================
    # Integration Tests — RAG Pipeline
    # ============================================================
    {
        "id": "PYT-15",
        "name": "Integration — RAG Pipeline",
        "args": [
            "tests/integration/test_rag_integration.py",
            "-v", "--tb=short"
        ]
    },

    # ============================================================
    # Integration Tests — Phase 11 Storage
    # ============================================================
    {
        "id": "PYT-16",
        "name": "Integration — Phase 11 Database & Storage",
        "args": [
            "tests/integration/test_database_storage.py",
            "-v", "--tb=short"
        ]
    },

    # ============================================================
    # Integration Tests — Phase 12 Services & Security
    # ============================================================
    {
        "id": "PYT-17",
        "name": "Integration — Phase 12 Services & Security",
        "args": [
            "tests/integration/test_service_security_integration.py",
            "-v", "--tb=short"
        ]
    },
]

print(f"Import checks defined : {len(IMPORT_TESTS)}")
print(f"Pytest suites defined : {len(PYTEST_TESTS)}")
print(f"Total checks          : {len(IMPORT_TESTS) + len(PYTEST_TESTS)}")

Import checks defined : 13
Pytest suites defined : 17
Total checks          : 30


## 2. Run all checks

A check is marked **PASSED** only when the import code finishes without an exception or pytest exits with code `0`. Missing test files, missing dependencies, collection errors, assertion failures, and runtime exceptions are reported separately with their details.

In [3]:

# ============================================================
# 1. Validate pytest configuration before execution
# ============================================================

all_test_files = []

for test in PYTEST_TESTS:
    for arg in test["args"]:
        if arg.startswith("tests/") and arg.endswith(".py"):
            all_test_files.append(arg)

# Find all actual test files in the project
actual_test_files = sorted(
    str(path.relative_to(PROJECT_ROOT))
    for path in (PROJECT_ROOT / "tests").rglob("test_*.py")
)

# Missing paths in configuration
missing_files = [
    path for path in all_test_files
    if not (PROJECT_ROOT / path).is_file()
]

# Duplicate assignments
file_counts = Counter(all_test_files)

duplicates = {
    path: count
    for path, count in file_counts.items()
    if count > 1
}

# Test files not included in configuration
unassigned_files = sorted(
    set(actual_test_files) - set(all_test_files)
)

print("=" * 65)
print("PYTEST CONFIGURATION AUDIT")
print("=" * 65)

print("Configured test files:", len(all_test_files))
print("Unique configured files:", len(set(all_test_files)))
print("Actual test files:", len(actual_test_files))

print("\nMissing files:", missing_files)
print("Duplicate assignments:", duplicates)
print("Unassigned test files:", unassigned_files)

assert not missing_files, "Missing test paths detected."
assert not duplicates, "Duplicate test assignments detected."
assert not unassigned_files, "Some test files are not assigned to a group."

print("\nConfiguration validation: PASSED")


# ============================================================
# 2. Result storage
# ============================================================

results = []


# ============================================================
# 3. Import test runner
# ============================================================

def run_import_check(test):
    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()

    started = time.perf_counter()

    try:
        with contextlib.redirect_stdout(stdout_buffer), \
             contextlib.redirect_stderr(stderr_buffer):

            exec(
                compile(test["code"], f"<{test['id']}>", "exec"),
                {"__name__": "__main__"}
            )

        status = "PASSED"
        error_type = None
        error_message = None
        tb = None

    except Exception as exc:
        status = "FAILED"
        error_type = type(exc).__name__
        error_message = str(exc)
        tb = traceback.format_exc()

    return {
        "id": test["id"],
        "type": "IMPORT",
        "name": test["name"],
        "source": f"CODES.md:{test['source_line']}",
        "test_line": test["code"],
        "status": status,
        "duration_sec": round(time.perf_counter() - started, 3),
        "stdout": stdout_buffer.getvalue(),
        "stderr": stderr_buffer.getvalue(),
        "error_type": error_type,
        "error_message": error_message,
        "traceback": tb,
        "return_code": 0 if status == "PASSED" else None,
    }


# ============================================================
# 4. Pytest runner
# ============================================================

def run_pytest_check(test):
    started = time.perf_counter()

    command = [
        sys.executable,
        "-m",
        "pytest",
        *test["args"]
    ]

    try:
        completed = subprocess.run(
            command,
            cwd=PROJECT_ROOT,
            capture_output=True,
            text=True,
            timeout=1800,
            env=os.environ.copy()
        )

        status = "PASSED" if completed.returncode == 0 else "FAILED"

        stdout = completed.stdout
        stderr = completed.stderr
        rc = completed.returncode

        error_type = None if rc == 0 else "PytestFailure"

        error_message = (
            None if rc == 0
            else f"pytest exited with code {rc}."
        )

        tb = None

    except subprocess.TimeoutExpired as exc:
        status = "ERROR"
        rc = None

        stdout = (
            exc.stdout.decode(errors="replace")
            if isinstance(exc.stdout, bytes)
            else (exc.stdout or "")
        )

        stderr = (
            exc.stderr.decode(errors="replace")
            if isinstance(exc.stderr, bytes)
            else (exc.stderr or "")
        )

        error_type = "TimeoutExpired"
        error_message = "Test command exceeded 30 minutes."
        tb = traceback.format_exc()

    except Exception as exc:
        status = "ERROR"
        rc = None
        stdout = ""
        stderr = ""
        error_type = type(exc).__name__
        error_message = str(exc)
        tb = traceback.format_exc()

    source = "CODES.md:" + ", ".join(
        map(str, test.get("source_lines", []))
    )

    return {
        "id": test["id"],
        "type": "PYTEST",
        "name": test["name"],
        "source": source,
        "test_line": " ".join(command),
        "status": status,
        "duration_sec": round(time.perf_counter() - started, 3),
        "stdout": stdout,
        "stderr": stderr,
        "error_type": error_type,
        "error_message": error_message,
        "traceback": tb,
        "return_code": rc,
    }


# ============================================================
# 5. Execute import checks
# ============================================================

print("\n" + "=" * 65)
print("IMPORT CHECKS")
print("=" * 65)

for test in IMPORT_TESTS:
    result = run_import_check(test)
    results.append(result)

    print(
        f"{result['id']} | "
        f"{result['status']:<6} | "
        f"{result['name']}"
    )


# ============================================================
# 6. Execute pytest groups
# ============================================================

print("\n" + "=" * 65)
print("PYTEST GROUPS")
print("=" * 65)

for test in PYTEST_TESTS:
    result = run_pytest_check(test)
    results.append(result)

    print(
        f"{result['id']} | "
        f"{result['status']:<6} | "
        f"{result['name']} | "
        f"{result['duration_sec']} sec"
    )

print("\nAll configured checks have finished.")


PYTEST CONFIGURATION AUDIT
Configured test files: 37
Unique configured files: 37
Actual test files: 37

Missing files: []
Duplicate assignments: {}
Unassigned test files: []

Configuration validation: PASSED

IMPORT CHECKS
IMP-01 | PASSED | Settings configuration import
IMP-02 | PASSED | Storage, security, and service public imports
IMP-03 | PASSED | Database module import
IMP-04 | PASSED | SQLAlchemy dependency import
IMP-05 | PASSED | Database models import
IMP-06 | PASSED | Services and security imports
IMP-07 | PASSED | DocumentService public import
IMP-08 | PASSED | IngestionService module import
IMP-09 | PASSED | Services and security imports (repeat checkpoint)
IMP-10 | PASSED | ChatService import
IMP-11 | PASSED | RAGPipeline import
IMP-12 | PASSED | RAGError import
IMP-13 | PASSED | Core services module imports

PYTEST GROUPS
PYT-01 | FAILED | Phase 1 — Foundation & Configuration | 1.058 sec
PYT-02 | FAILED | Phase 2 — Domain Models | 1.006 sec
PYT-03 | FAILED | Phase 3 — Docu

## 3. Summary and line-level report

The summary counts checks, not source-file lines. Each report row corresponds to one distinct check; consolidated pytest rows list all source lines where equivalent variants appeared.

In [7]:
import pandas as pd
from IPython.display import display

report_df = pd.DataFrame([{
    "ID": r["id"],
    "Type": r["type"],
    "Test": r["name"],
    "Source line(s)": r["source"],
    "Status": r["status"],
    "Duration (s)": r["duration_sec"],
    "Return code": r["return_code"],
    "Error type": r["error_type"],
    "Error": r["error_message"],
} for r in results])

total = len(results)
passed = sum(r["status"] == "PASSED" for r in results)
failed = sum(r["status"] == "FAILED" for r in results)
errors = sum(r["status"] == "ERROR" for r in results)
skipped = sum(r["status"] == "SKIPPED" for r in results)

print("========== TEST EXECUTION SUMMARY ==========")
print(f"Total checks : {total}")
print(f"Passed       : {passed}")
print(f"Failed       : {failed}")
print(f"Errors       : {errors}")
print(f"Skipped      : {skipped}")
print(f"Success rate : {(passed / total * 100):.1f}%" if total else "Success rate : N/A")
print("============================================")
display(report_df)

========== TEST EXECUTION SUMMARY ==========
Total checks : 30
Passed       : 15
Failed       : 15
Errors       : 0
Skipped      : 0
Success rate : 50.0%


,ID,Type,Test,Source line(s),Status,Duration (s),Return code,Error type,Error
0,IMP-01,IMPORT,Settings configuration import,CODES.md:318,PASSED,0.205,0,NaN,NaN
1,IMP-02,IMPORT,"Storage, security, and service public imports",CODES.md:3396,PASSED,7.455,0,NaN,NaN
2,IMP-03,IMPORT,Database module import,CODES.md:3411,PASSED,0.000,0,NaN,NaN
3,IMP-04,IMPORT,SQLAlchemy dependency import,CODES.md:3415,PASSED,0.000,0,NaN,NaN
4,IMP-05,IMPORT,Database models import,CODES.md:3425,PASSED,0.000,0,NaN,NaN
5,IMP-06,IMPORT,Services and security imports,CODES.md:3433,PASSED,0.000,0,NaN,NaN
6,IMP-07,IMPORT,DocumentService public import,CODES.md:3443,PASSED,0.000,0,NaN,NaN
7,IMP-08,IMPORT,IngestionService module import,CODES.md:3447,PASSED,0.000,0,NaN,NaN
8,IMP-09,IMPORT,Services and security imports (repeat checkpoint),CODES.md:3661,PASSED,0.000,0,NaN,NaN
9,IMP-10,IMPORT,ChatService import,CODES.md:3669,PASSED,0.000,0,NaN,NaN


## 4. Inspect failures in detail

This displays the exact code/command, captured output, exception type, message, and traceback for every failed or errored check.

In [8]:
problem_results = [r for r in results if r["status"] in {"FAILED", "ERROR"}]

if not problem_results:
    print("No failed or errored checks.")
else:
    for r in problem_results:
        print("\\n" + "=" * 100)
        print(f"{r['id']} — {r['name']} — {r['status']}")
        print(f"Source: {r['source']}")
        print("\\nTEST LINE / COMMAND:")
        print(r["test_line"])
        print(f"\\nERROR TYPE: {r['error_type']}")
        print(f"ERROR MESSAGE: {r['error_message']}")
        if r["return_code"] is not None:
            print(f"RETURN CODE: {r['return_code']}")
        if r["stdout"]:
            print("\\n--- STDOUT (last 12000 chars) ---")
            print(r["stdout"][-12000:])
        if r["stderr"]:
            print("\\n--- STDERR (last 12000 chars) ---")
            print(r["stderr"][-12000:])
        if r["traceback"]:
            print("\\n--- TRACEBACK ---")
            print(r["traceback"])

\n====================================================================================================
PYT-01 — Phase 1 — Foundation & Configuration — FAILED
Source: CODES.md:
\nTEST LINE / COMMAND:
/home/ahmed/anaconda3/envs/genai/bin/python -m pytest tests/unit/test_settings.py tests/unit/test_file_utils.py tests/unit/test_text_utils.py -v --tb=short
\nERROR TYPE: PytestFailure
ERROR MESSAGE: pytest exited with code 1.
RETURN CODE: 1
\n--- STDOUT (last 12000 chars) ---
============================= test session starts ==============================
platform linux -- Python 3.11.17, pytest-9.1.1, pluggy-1.7.0
rootdir: /run/media/ahmed/Data/RAG/Egyptian-Legal-AI
configfile: pyproject.toml
plugins: asyncio-1.4.0, langsmith-0.14.11, anyio-4.15.1, platformdirs-4.12.3
asyncio: mode=Mode.AUTO, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 32 items

tests/unit/test_settings.py .......F....                                 [ 37%]
tests

## 5. Export results

Exports the summary and full raw output/tracebacks to the project `reports/` directory. The JSON file is useful for later comparison between runs.

In [6]:
from pathlib import Path
import csv

report_dir = PROJECT_ROOT / "reports" / "code_audit"
report_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

summary_path = report_dir / f"audit_summary_{timestamp}.csv"
details_path = report_dir / f"audit_details_{timestamp}.json"

report_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
with details_path.open("w", encoding="utf-8") as f:
    json.dump({
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "project_root": str(PROJECT_ROOT),
        "python_executable": sys.executable,
        "summary": {
            "total": total, "passed": passed, "failed": failed,
            "errors": errors, "skipped": skipped,
            "success_rate_percent": round(passed / total * 100, 2) if total else None,
        },
        "results": results,
    }, f, ensure_ascii=False, indent=2, default=str)

print(f"CSV summary : {summary_path}")
print(f"JSON details: {details_path}")

CSV summary : /run/media/ahmed/Data/RAG/Egyptian-Legal-AI/reports/code_audit/audit_summary_20261104_092722.csv
JSON details: /run/media/ahmed/Data/RAG/Egyptian-Legal-AI/reports/code_audit/audit_details_20261104_092722.json


In [2]:
LIST = [
"src/workers/__init__.py",
"src/workers/job_status.py",
"src/workers/job_store.py",
"src/workers/job_manager.py",
"src/workers/ingestion_worker.py",
"workers/__init__.py",
"workers/ingestion_worker.py",
"api/schemas/jobs.py",
"api/routes/jobs.py",
"src/services/factory.py"
]

for file in LIST:
    print(f"touch '{file}'")

touch 'src/workers/__init__.py'
touch 'src/workers/job_status.py'
touch 'src/workers/job_store.py'
touch 'src/workers/job_manager.py'
touch 'src/workers/ingestion_worker.py'
touch 'workers/__init__.py'
touch 'workers/ingestion_worker.py'
touch 'api/schemas/jobs.py'
touch 'api/routes/jobs.py'
touch 'src/services/factory.py'


In [ ]:
# %% [markdown]
# # Preprocessing: Split CSV Articles
# Split rows containing multiple "مواد" into one "مادة" per row.

# %% [markdown]
# ## 1. Imports & Setup

# %%
import pandas as pd
import re
from pathlib import Path
import sys

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

# Reuse the existing article pattern from the project
from src.preprocessing.patterns import LEGAL_ARTICLE_PATTERN

# %% [markdown]
# ## 2. Load the CSV

# %%
CSV_PATH = Path("data/raw/official_laws/sample.csv")
df = pd.read_csv(CSV_PATH, encoding="utf-8")

print(f"Original shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

# %% [markdown]
# ## 3. Identify the Text Column
# Adjust `TEXT_COLUMN` if your column has a different name.

# %%
# Auto-detect the text column (the one with the longest average content)
TEXT_COLUMN = None
max_avg_len = 0
for col in df.columns:
    if df[col].dtype == object:
        avg_len = df[col].dropna().astype(str).str.len().mean()
        if avg_len > max_avg_len:
            max_avg_len = avg_len
            TEXT_COLUMN = col

print(f"Detected text column: '{TEXT_COLUMN}' (avg length: {max_avg_len:.0f} chars)")

# %% [markdown]
# ## 4. Article Splitting Function
# Uses the same regex pattern as `LegalStructureParser` to detect article boundaries.

# %%
def split_articles(text: str) -> list[str]:
    """
    Split a text containing multiple articles into individual article segments.
    
    Splits at boundaries matching: المادة رقم 1 | مادة 2 | المادة الأولى | مادة رقم 5
    Preserves the article header in each resulting segment.
    """
    if not text or not isinstance(text, str) or not text.strip():
        return []
    
    # Regex: split right BEFORE each article marker, keeping the marker with the text
    # Handles: المادة 1, مادة رقم 2, المادة الأولى, ماده 3, المادة 12 مكرر
    article_boundary = re.compile(
        r'(?=(?:^|\n)\s*(?:المادة|مادة|ماده)\s*(?:رقم\s*)?'
        r'(?:\d+|[٠-٩]+|الأولى|الثانية|الثالثة|الرابعة|الخامسة|السادسة|السابعة|الثامنة|التاسعة|العاشرة)'
        r'(?:\s*(?:مكرر|مكررة))?)',
        re.MULTILINE | re.IGNORECASE
    )
    
    parts = article_boundary.split(text)
    
    # Filter empty segments and strip whitespace
    articles = [part.strip() for part in parts if part and part.strip()]
    return articles


# Quick test
test_text = "المادة رقم 1: هذا نص المادة الأولى.\nالمادة رقم 2: هذا نص المادة الثانية.\nالمادة رقم 3: هذا نص المادة الثالثة."
result = split_articles(test_text)
print(f"Test split: {len(result)} articles found")
for i, art in enumerate(result):
    print(f"  [{i}] {art[:80]}...")

# %% [markdown]
# ## 5. Apply Splitting to All Rows

# %%
new_rows = []
split_count = 0
total_original = len(df)

for idx, row in df.iterrows():
    text = str(row[TEXT_COLUMN]) if pd.notna(row[TEXT_COLUMN]) else ""
    articles = split_articles(text)
    
    if len(articles) <= 1:
        # No split needed — keep original row as-is
        new_rows.append(row.to_dict())
    else:
        # Split into multiple rows
        split_count += 1
        for article_idx, article_text in enumerate(articles):
            new_row = row.to_dict()
            new_row[TEXT_COLUMN] = article_text
            new_row["article_part_index"] = article_idx  # Track which part
            new_rows.append(new_row)

df_split = pd.DataFrame(new_rows)

print(f"Original rows: {total_original}")
print(f"Rows that were split: {split_count}")
print(f"New total rows: {len(df_split)}")
print(f"Average articles per split row: {(len(df_split) - total_original + split_count) / max(split_count, 1):.2f}")

# %% [markdown]
# ## 6. Verify Results

# %%
# Show before/after comparison for a split row
sample_split_row = df_split[df_split["article_part_index"] == 0].iloc[0] if "article_part_index" in df_split.columns else None
if sample_split_row is not None:
    print("=== Sample split article ===")
    print(sample_split_row[TEXT_COLUMN][:300])
    print(f"\n=== Chunk size: {len(sample_split_row[TEXT_COLUMN])} chars ===")

# %%
# Check chunk size distribution
df_split["text_length"] = df_split[TEXT_COLUMN].astype(str).str.len()

print(f"\n--- Text Length Statistics (after split) ---")
print(f"  Min:    {df_split['text_length'].min()} chars")
print(f"  Mean:   {df_split['text_length'].mean():.0f} chars")
print(f"  Median: {df_split['text_length'].median():.0f} chars")
print(f"  Max:    {df_split['text_length'].max()} chars")

# %% [markdown]
# ## 7. Save the New CSV

# %%
OUTPUT_PATH = Path("data/raw/official_laws/sample_split.csv")
df_split.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"✅ Saved to: {OUTPUT_PATH}")
print(f"   Rows: {len(df_split)}")

# %% [markdown]
# ## 8. Comparison: Before vs After

# %%
print("=" * 60)
print(f"{'Metric':<30} {'Before':<15} {'After':<15}")
print("=" * 60)
print(f"{'Total rows':<30} {len(df):<15} {len(df_split):<15}")
print(f"{'Avg text length (chars)':<30} {df[TEXT_COLUMN].astype(str).str.len().mean():<15.0f} {df_split['text_length'].mean():<15.0f}")
print(f"{'Max text length (chars)':<30} {df[TEXT_COLUMN].astype(str).str.len().max():<15} {df_split['text_length'].max():<15}")
print(f"{'Min text length (chars)':<30} {df[TEXT_COLUMN].astype(str).str.len().min():<15} {df_split['text_length'].min():<15}")
print("=" * 60)



In [1]:
import pandas as pd

In [15]:
df = pd.read_csv("../data/raw/official_laws/split_egypt_legal_corpus.csv")
old = pd.read_csv("../data/raw/official_laws/egypt_legal_corpus.csv")

In [16]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
tokens,72902.0,55926.196017,50668.153809,112.0,13714.0,41522.0,91938.0,235109.0
segment_index,72902.0,115.776755,168.678562,0.0,14.0,50.0,150.0,1157.0
segment_count,72902.0,232.553510,268.024537,1.0,43.0,138.0,339.0,1158.0
char_count,72902.0,502.828866,1041.068085,21.0,113.0,243.0,518.0,57764.0


In [17]:
old.describe().T

,count,mean,std,min,25%,50%,75%,max
tokens,2434.0,10293.497124,22272.716059,112.0,714.0,2357.5,8540.0,235109.0


In [21]:
for i in range(5):
    print(df[df['char_count']>1000]['text'].iloc[i])
    print("\n" + "-"*80 + "\n")

مادة 1 تجب النفقة للزوجة علي زوجها من تاريخ العقد الصحيح اذا سلمت نفسها الية لو حكما حتي لو كانت موسرة او مختلفة معه في الدين ولا يمنع مرض الزوجة من استحقاقها للنفقة وتشمل النفقة الغذاء والكسوة والسكن ومصاريف العلاج وغير ذلك بما يقتضي به الشرع . ولا تجب النفقة للزوجة اذا ارتدت ، او امتنعت مختارة عن تسليم نفسها دون حق ، او اضطرت الي بسبب ليس من قبل الزوج او خرجت دون اذن زوجها . ولا يعتبر سبب لسقوط نفقة الزوجة خروجها من مسكن الزوجية دون اذن زوجها في الاحوال التي تباح فيها ذلك بحكم الشرع مما ورد به نص او جري به عرف او قضت به ضرورة . ولا خروجها للعمل المشروع ما لم يظهر ان استعمالها لهذا الحق المشروط منسوب باساءة استعمال الحق . او مناف لمصلحة الاسرة وطلب منها الزوج الامتناع عنه . وتعتبر نفقة الزوجة دينا علي الزوج من تاريخ امتناعة عن الانفاق مع وجوبة ، ولا تقسط الا بالاداء او الابراء . ولا تسمع دعوي النفقة عن مدة ماضية لاكثر من سنة نهايتها تاريخ رفع الدعوي . ولا يقبل من الزوج التمسك بالمقاصة بين نفقة الزوجة وبين دين له عليها الا فيما يزيد علي ما يفي بحاجتها الضرورية ، ويكون له من نفقة الزوجة